# Praktikum Komputasi 1: Pengantar Simulasi Numerik (Julia)

Selamat datang di modul simulasi komputasi untuk mata kuliah Persamaan Diferensial. Pada dokumen ini, kita menggunakan bahasa pemrograman **Julia**. 

Kita akan mensimulasikan respons transien rangkaian RL seri. Persamaan Diferensial Biasa (PDB) untuk arus $i(t)$ pada rangkaian ini adalah:
$$ L \frac{di}{dt} + R i = V_0 $$
$$ \frac{di}{dt} = \frac{V_0 - R i}{L} $$

Alih-alih langsung menggunakan persamaan analitik (eksponensial), kita akan melatih insting teknik (engineering) kita dengan **Metode Numerik Euler**. Komputer akan menyelesaikan persamaan diferensial tersebut secara "kasar" namun efektif.

In [ ]:
# Jalankan sel ini terlebih dahulu untuk memuat pustaka visualisasi
using Plots

## 1. Persiapan Parameter Sistem
Tentukan parameter fisik dari induktor, resistor, dan sumber tegangan.

In [ ]:
R = 100.0   # Resistansi dalam Ohm
L = 0.1     # Induktansi dalam Henry
V₀ = 5.0    # Sumber tegangan DC 5 Volt

# Durasi simulasi
t_akhir = 0.005 # 5 milidetik
Δt = 0.0001    # Ukuran langkah waktu (step size)

## 2. Metode Numerik Euler
Metode Euler memperkirakan nilai masa depan berdasarkan nilai masa kini ditambah laju perubahan saat ini dikali rentang waktu:
$$ i(t + \Delta t) \approx i(t) + \Delta t \cdot \left( \frac{di}{dt} \right) $$

In [ ]:
waktu = 0.0:Δt:t_akhir
N = length(waktu)

arus_numerik = zeros(N)
arus_numerik[1] = 0.0 # Syarat awal: i(0) = 0

for k in 1:(N-1)
    # Hitung laju perubahan saat ini (di/dt)
    di_dt = (V₀ - R * arus_numerik[k]) / L
    
    # Prediksi arus di masa depan (Euler)
    arus_numerik[k+1] = arus_numerik[k] + Δt * di_dt
end


## 3. Validasi dengan Solusi Analitik Eksak
Mari kita bandingkan hasil pendekatan *coding* kita dengan hasil turunan matematika (Solusi Khusus PDB) yang telah diajarkan di kelas:
$$ i(t) = \frac{V_0}{R} \left( 1 - e^{-\frac{R}{L}t} \right) $$

In [ ]:
arus_eksak = (V₀/R) .* (1 .- exp.(-(R/L) .* waktu))

plot(waktu .* 1000, arus_eksak .* 1000, label="Solusi Analitik Eksak", lw=3, color=:blue, xlabel="Waktu (ms)", ylabel="Arus (mA)")
plot!(waktu .* 1000, arus_numerik .* 1000, label="Pendekatan Numerik Euler", ls=:dash, lw=3, color=:orange)
title!("Perbandingan Arus Induktor")

**Tugas Diskusi:** 
1. Apa yang terjadi jika nilai `Δt` (di sel parameter) diperbesar menjadi `0.002`? Jelaskan mengapa kurva warna oranye tidak lagi akurat!
2. Ubahlah nilai `R = 0`. Apa yang terjadi pada arus? Jelaskan secara fisika!

## 4. [Opsional] Tingkat Lanjut: Pustaka Standar Industri (SciML)
Di dunia industri, kita tidak menulis perulangan `for` secara manual. Julia memiliki pustaka ekosistem **DifferentialEquations.jl** yang sangat cepat untuk ini.

In [ ]:
using DifferentialEquations

f(u, p, t) = (V₀ - R*u) / L
u0 = 0.0
tspan = (0.0, 0.005)

prob = ODEProblem(f, u0, tspan)
sol = solve(prob, Tsit5())

plot(sol, label="SciML Tsit5()", lw=2, color=:green, xlabel="Waktu (s)", ylabel="Arus (A)")